In [ ]:
# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # Alternatif: gunakan AdamW dari PyTorch

# Transformers (HuggingFace)
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Alternatif jika tetap ingin pakai AdamW dari transformers:
# from transformers.optimization import AdamW

# Data Handling
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Utility
import re
import string
from tqdm.auto import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
url = 'https://raw.githubusercontent.com/syahrulazka/indihome-tweet-analytics/refs/heads/main/indihome_tweet_data_for_training.csv'
df = pd.read_csv(url)

display(df.head())

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username,Sentiment
0,1936938377678610742,2025-06-23 12:47:33+00:00,0,@IndiHome Adaptor sudah kencang. Prosedur awal...,1937130406748569707,NaN,IndiHome,in,NaN,0,1,0,https://x.com/undefined/status/193713040674856...,254827832,NaN,Negatif
1,1937117284558073864,2025-06-23 12:47:50+00:00,0,@aulia_ak22 Huhu maafin Kakk Kurang optimalnya...,1937130475929358622,NaN,aulia_ak22,in,NaN,0,0,0,https://x.com/undefined/status/193713047592935...,1075673058931806208,NaN,Netral
2,1937122719151345948,2025-06-23 12:48:33+00:00,0,@IndiHome Cek dm ya @IndiHome,1937130656959778975,NaN,IndiHome,in,NaN,0,0,0,https://x.com/undefined/status/193713065695977...,753400263826771968,NaN,Netral
3,1937117822993477978,2025-06-23 12:49:57+00:00,0,@mochhhh_ Malam Kak maaf ya atas kendala yg di...,1937131007876243480,NaN,mochhhh_,in,NaN,0,0,0,https://x.com/undefined/status/193713100787624...,1075673058931806208,NaN,Netral
4,1937118185867887069,2025-06-23 12:51:29+00:00,0,@co_je85499 Kak maafin ya jadi terganggu aktiv...,1937131397422141604,NaN,co_je85499,in,NaN,0,0,0,https://x.com/undefined/status/193713139742214...,1075673058931806208,NaN,Netral


In [ ]:
df['Sentiment'].value_counts()

,count
Sentiment,
Netral,60648
Negatif,27288
Positif,3323


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
class TextPreprocessor:
    def __init__(self):
        pass

    def clean_text(self, text):
        """Clean and preprocess text"""
        if pd.isna(text):
            return ""

        # Convert to lowercase
        text = text.lower()

        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

        # Remove user mentions and hashtags
        text = re.sub(r'@\w+|#\w+', '', text)

        # Remove extra whitespaces
        text = re.sub(r'\s+', ' ', text)

        # Remove punctuation (optional - you might want to keep some)
        # text = text.translate(str.maketrans('', '', string.punctuation))

        # Strip whitespace
        text = text.strip()

        return text

    def preprocess_data(self, df, text_column, label_column):
        """Preprocess the entire dataset"""
        # Clean texts
        df[text_column] = df[text_column].apply(self.clean_text)

        # Remove empty texts
        df = df[df[text_column].str.len() > 0]

        # Map labels to integers if they're strings
        if df[label_column].dtype == 'object':
            label_mapping = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
            df[label_column] = df[label_column].map(label_mapping)

        return df

In [ ]:
class BERTSentimentClassifier:
    def __init__(self, model_name='bert-base-uncased', num_classes=3, max_length=128):
        self.model_name = model_name
        self.num_classes = num_classes
        self.max_length = max_length
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        print(f"Using device: {self.device}")

        # Initialize tokenizer and model
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_classes
        ).to(self.device)

        self.preprocessor = TextPreprocessor()

    def prepare_data(self, df, text_column='text', label_column='label', test_size=0.2, val_size=0.1):
        """Prepare and split data"""
        # Preprocess data
        df = self.preprocessor.preprocess_data(df, text_column, label_column)

        # Split data
        X = df[text_column].values
        y = df[label_column].values

        # First split: train+val and test
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )

        # Second split: train and val
        val_size_adjusted = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_size_adjusted, random_state=42, stratify=y_temp
        )

        print(f"Train size: {len(X_train)}")
        print(f"Validation size: {len(X_val)}")
        print(f"Test size: {len(X_test)}")

        # Create datasets
        train_dataset = SentimentDataset(X_train, y_train, self.tokenizer, self.max_length)
        val_dataset = SentimentDataset(X_val, y_val, self.tokenizer, self.max_length)
        test_dataset = SentimentDataset(X_test, y_test, self.tokenizer, self.max_length)

        return train_dataset, val_dataset, test_dataset

    def create_data_loaders(self, train_dataset, val_dataset, test_dataset, batch_size=16):
        """Create data loaders"""
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        return train_loader, val_loader, test_loader

    def train(self, train_loader, val_loader, epochs=3, learning_rate=2e-5):
        """Train the model"""
        # Optimizer and scheduler
        optimizer = AdamW(self.model.parameters(), lr=learning_rate)
        total_steps = len(train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=0,
            num_training_steps=total_steps
        )

        best_val_accuracy = 0
        train_losses = []
        val_accuracies = []

        for epoch in range(epochs):
            print(f'\nEpoch {epoch + 1}/{epochs}')
            print('-' * 30)

            # Training phase
            self.model.train()
            total_train_loss = 0

            train_pbar = tqdm(train_loader, desc='Training')
            for batch in train_pbar:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                optimizer.zero_grad()

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss
                total_train_loss += loss.item()

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

                train_pbar.set_postfix({'loss': loss.item()})

            avg_train_loss = total_train_loss / len(train_loader)
            train_losses.append(avg_train_loss)

            # Validation phase
            val_accuracy = self.evaluate(val_loader)
            val_accuracies.append(val_accuracy)

            print(f'Train Loss: {avg_train_loss:.4f}')
            print(f'Val Accuracy: {val_accuracy:.4f}')

            # Save best model
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                self.save_model('best_model')
                print('New best model saved!')

        return train_losses, val_accuracies

    def evaluate(self, data_loader):
        """Evaluate the model"""
        self.model.eval()
        predictions = []
        true_labels = []

        with torch.no_grad():
            for batch in tqdm(data_loader, desc='Evaluating'):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                _, preds = torch.max(outputs.logits, dim=1)
                predictions.extend(preds.cpu().tolist())
                true_labels.extend(labels.cpu().tolist())

        accuracy = accuracy_score(true_labels, predictions)
        return accuracy

    def detailed_evaluation(self, data_loader):
        """Get detailed evaluation metrics"""
        self.model.eval()
        predictions = []
        true_labels = []

        with torch.no_grad():
            for batch in tqdm(data_loader, desc='Evaluating'):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                _, preds = torch.max(outputs.logits, dim=1)
                predictions.extend(preds.cpu().tolist())
                true_labels.extend(labels.cpu().tolist())

        accuracy = accuracy_score(true_labels, predictions)
        report = classification_report(
            true_labels, predictions,
            target_names=['Negative', 'Neutral', 'Positive']
        )
        cm = confusion_matrix(true_labels, predictions)

        return accuracy, report, cm

    def predict(self, text):
        """Predict sentiment for a single text"""
        self.model.eval()

        # Preprocess text
        text = self.preprocessor.clean_text(text)

        # Tokenize
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].to(self.device)
        attention_mask = encoding['attention_mask'].to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
            predicted_class = torch.argmax(predictions, dim=-1).item()
            confidence = predictions[0][predicted_class].item()

        label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

        return {
            'predicted_label': label_map[predicted_class],
            'confidence': confidence,
            'probabilities': {
                'negative': predictions[0][0].item(),
                'neutral': predictions[0][1].item(),
                'positive': predictions[0][2].item()
            }
        }

    def save_model(self, path):
        """Save model and tokenizer"""
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"Model saved to {path}")

    def load_model(self, path):
        """Load model and tokenizer"""
        self.model = BertForSequenceClassification.from_pretrained(path).to(self.device)
        self.tokenizer = BertTokenizer.from_pretrained(path)
        print(f"Model loaded from {path}")

In [ ]:
def main():
    # df = pd.read_csv('your_dataset.csv')  # Replace with your dataset

    # Initialize classifier
    classifier = BERTSentimentClassifier(
        model_name='bert-base-uncased',
        num_classes=3,
        max_length=128
    )

    # Prepare data
    train_dataset, val_dataset, test_dataset = classifier.prepare_data(
        df, text_column='full_text', label_column='Sentiment'
    )

    # Create data loaders
    train_loader, val_loader, test_loader = classifier.create_data_loaders(
        train_dataset, val_dataset, test_dataset, batch_size=16
    )

    # Train the model
    print("Starting training...")
    train_losses, val_accuracies = classifier.train(
        train_loader, val_loader, epochs=3, learning_rate=2e-5
    )

    # Load best model for evaluation
    classifier.load_model('best_model')

    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_accuracy, test_report, test_cm = classifier.detailed_evaluation(test_loader)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print("\nClassification Report:")
    print(test_report)
    print("\nConfusion Matrix:")
    print(test_cm)

    # Test prediction
    sample_texts = [
        "Indihome lemott bangett dahhh",
        "tumben indihome lancar",
        "min, indihome kenapa ya?"
    ]

    print("\nSample predictions:")
    for text in sample_texts:
        result = classifier.predict(text)
        print(f"Text: '{text}'")
        print(f"Prediction: {result['predicted_label']} (confidence: {result['confidence']:.4f})")
        print()

if __name__ == "__main__":
    main()

Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Train size: 63881
Validation size: 9126
Test size: 18252
Starting training...

Epoch 1/3
------------------------------


Training:   0%|          | 0/3993 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/571 [00:00<?, ?it/s]

...

Training:   0%|          | 0/3993 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/571 [00:00<?, ?it/s]

Train Loss: 0.1519
Val Accuracy: 0.9195
Model loaded from best_model

Evaluating on test set...


Evaluating:   0%|          | 0/1141 [00:00<?, ?it/s]

Test Accuracy: 0.9184

Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.88      0.89      5458
     Neutral       0.93      0.96      0.94     12130
    Positive       0.88      0.54      0.67       664

    accuracy                           0.92     18252
   macro avg       0.90      0.79      0.83     18252
weighted avg       0.92      0.92      0.92     18252


Confusion Matrix:
[[ 4813   640     5]
 [  492 11593    45]
 [   98   210   356]]

Sample predictions:
Text: 'Indihome lemott bangett dahhh'
Prediction: negative (confidence: 0.9993)

Text: 'tumben indihome lancar'
Prediction: positive (confidence: 0.9704)

Text: 'min, indihome kenapa ya?'
Prediction: neutral (confidence: 0.9826)

